# OpenMontage Trinity Pro — Notebook Maestro Multi-Experto

Notebook Colab Pro (A100 40/80GB) que expone 3 modelos expertos open-source como un mini-servidor HTTP consumido por `providers/trinity_provider.py` en Guion_expert.

**Los 3 expertos:**
| Experto | Modelo | Modalidad | Especialidad |
|---|---|---|---|
| 🎬 Cinematografía | Wan 2.1 14B | I2V | Paisajes, drones, planos lentos (MoE, físicas de mundo) |
| 🎭 Humanos | SkyReels V1 | T2V | Expresiones y matices emocionales (10M clips cine/TV) |
| 🔥 Macro/Física | HunyuanVideo 13B | T2V | Fuego, fluidos, colisiones (entrenamiento end-to-end) |

**Flujo:** el notebook carga UN modelo a la vez (purga VRAM entre llamadas), expone FastAPI + ngrok, devuelve mp4 al cliente.

**Endpoints:**
- `POST /i2v`  body: `{ image_b64, prompt, model, duration_seconds, aspect_ratio }` → `{ video_b64, elapsed_s, model_used }`
- `POST /t2v`  body: `{ prompt, model, duration_seconds, aspect_ratio }` → `{ video_b64, elapsed_s, model_used }`
- `GET  /health` → `{ status: "ok", models_loaded: [...], vram_mb: ... }`

**Auth:** header `X-Trinity-Token: <shared secret>`.

## Celda 1 · Setup global
Instala dependencias, autentica Hugging Face, define el token compartido con Guion_expert.

In [ ]:
import subprocess, sys, os

# Deps base
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers>=0.31.0", "transformers>=4.44.0",
                "accelerate>=0.34", "safetensors", "imageio[ffmpeg]",
                "sentencepiece", "einops", "protobuf",
                "fastapi", "uvicorn", "pyngrok", "python-multipart"],
               check=True)

# Verificar GPU
import torch
assert torch.cuda.is_available(), "No hay GPU disponible. Runtime → Change runtime type → A100"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

# Auth HF (opcional pero recomendado para Hunyuan gated)
from huggingface_hub import login
HF_TOKEN = os.getenv("HF_TOKEN") or "hf_REEMPLAZAR"
if HF_TOKEN and not HF_TOKEN.endswith("REEMPLAZAR"):
    login(token=HF_TOKEN)
    print("HF auth OK")
else:
    print("HF_TOKEN no seteado — OK si todos los modelos son públicos")

# Shared secret con Guion_expert (debe coincidir con TRINITY_TOKEN del .env)
import secrets
TRINITY_TOKEN = os.getenv("TRINITY_TOKEN") or secrets.token_hex(32)
print(f"TRINITY_TOKEN (copiá esto a .env de Guion_expert):\n  TRINITY_TOKEN={TRINITY_TOKEN}")

## Celda 2 · Gestor de VRAM + registry de pipelines

Gestor que carga un pipeline a demanda y purga el anterior para no saturar los 40/80GB de la A100.

In [ ]:
import gc, torch, time
from typing import Optional

class PipelineManager:
    """Gestiona un único pipeline residente en VRAM a la vez."""
    def __init__(self):
        self._current_key: Optional[str] = None
        self._current_pipe = None

    def _purge(self):
        if self._current_pipe is not None:
            del self._current_pipe
            self._current_pipe = None
            self._current_key = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    def get(self, key: str, loader):
        if self._current_key != key:
            print(f"[VRAM] purging '{self._current_key}' → loading '{key}'")
            self._purge()
            t0 = time.time()
            self._current_pipe = loader()
            print(f"[VRAM] loaded '{key}' in {time.time()-t0:.1f}s. Peak: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
            self._current_key = key
        return self._current_pipe

    def vram_mb(self) -> int:
        return int(torch.cuda.memory_allocated() / 1e6)

    def current(self) -> Optional[str]:
        return self._current_key

MANAGER = PipelineManager()
print("PipelineManager OK")

## Celda 3 · Loaders por modelo (Wan 2.1, SkyReels, Hunyuan)

Cada loader define cómo instanciar el pipeline. Solo se ejecuta cuando llega una request.

In [ ]:
import torch

def load_wan_i2v():
    """Wan 2.1 14B — Image-to-Video. MoE, excelente en paisajes/drones."""
    from diffusers import WanImageToVideoPipeline  # diffusers >= 0.31
    pipe = WanImageToVideoPipeline.from_pretrained(
        "Wan-AI/Wan2.1-I2V-14B-720P-Diffusers",
        torch_dtype=torch.bfloat16,
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    return pipe

def load_skyreels_t2v():
    """SkyReels V1 — Text-to-Video sobre Hunyuan. Humanos/expresiones."""
    from diffusers import HunyuanVideoPipeline
    pipe = HunyuanVideoPipeline.from_pretrained(
        "Skywork/SkyReels-V1-Hunyuan-T2V",
        torch_dtype=torch.bfloat16,
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    return pipe

def load_hunyuan_t2v():
    """HunyuanVideo base 13B — Text-to-Video. Física end-to-end, VFX, macro."""
    from diffusers import HunyuanVideoPipeline
    pipe = HunyuanVideoPipeline.from_pretrained(
        "tencent/HunyuanVideo",
        torch_dtype=torch.bfloat16,
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    return pipe

LOADERS = {
    "wan-2.1-14b":  ("i2v", load_wan_i2v),
    "wan-2.1":      ("i2v", load_wan_i2v),
    "skyreels-v1":  ("t2v", load_skyreels_t2v),
    "skyreels":     ("t2v", load_skyreels_t2v),
    "hunyuan-13b":  ("t2v", load_hunyuan_t2v),
    "hunyuan":      ("t2v", load_hunyuan_t2v),
}
print("Loaders registrados:", list(LOADERS.keys()))

## Celda 4 · Funciones de generación (I2V y T2V)

Wrapper que normaliza el input/output para todos los modelos.

In [ ]:
import base64, io, tempfile, time
from PIL import Image
import imageio

FPS = 24

def _ratio_to_size(aspect_ratio: str, base_long: int = 1024) -> tuple[int, int]:
    """Convierte '16:9' / '9:16' a (width, height)."""
    mapping = {
        "16:9":  (1280, 720),
        "9:16":  (720, 1280),
        "1:1":   (960, 960),
        "4:3":   (1024, 768),
        "21:9":  (1344, 576),
    }
    return mapping.get(aspect_ratio, (1280, 720))

def _frames_to_mp4(frames, fps: int = FPS) -> bytes:
    """Lista de frames PIL/np → mp4 bytes."""
    tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    writer = imageio.get_writer(tmp.name, fps=fps, codec="libx264", quality=8)
    for f in frames:
        if hasattr(f, "convert"):
            f = f.convert("RGB")
        writer.append_data(np.array(f) if hasattr(f, "size") else f)
    writer.close()
    with open(tmp.name, "rb") as fh:
        return fh.read()

import numpy as np  # imported late to match import order

def run_i2v(model: str, image_b64: str, prompt: str,
            duration_seconds: float, aspect_ratio: str) -> dict:
    """Ejecuta I2V. Solo Wan 2.1 está soportado hoy."""
    t0 = time.time()
    loader_key, loader_fn = LOADERS.get(model, (None, None))
    if loader_key != "i2v":
        raise ValueError(f"Modelo I2V no soportado: {model}")

    pipe = MANAGER.get(model, loader_fn)
    image = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
    width, height = _ratio_to_size(aspect_ratio)
    image = image.resize((width, height))

    num_frames = int(round(duration_seconds * FPS))
    num_frames = max(24, min(num_frames, 240))  # 1s ≤ duración ≤ 10s en bf16

    out = pipe(
        image=image,
        prompt=prompt,
        num_frames=num_frames,
        height=height,
        width=width,
        guidance_scale=5.0,
    )
    frames = out.frames[0]  # diffusers devuelve [batch][frames]
    video_bytes = _frames_to_mp4(frames, fps=FPS)
    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
    return {
        "video_b64": video_b64,
        "elapsed_s": round(time.time() - t0, 1),
        "model_used": model,
        "vram_peak_mb": int(torch.cuda.max_memory_allocated() / 1e6),
    }

def run_t2v(model: str, prompt: str,
            duration_seconds: float, aspect_ratio: str) -> dict:
    """Ejecuta T2V (SkyReels o Hunyuan)."""
    t0 = time.time()
    loader_key, loader_fn = LOADERS.get(model, (None, None))
    if loader_key != "t2v":
        raise ValueError(f"Modelo T2V no soportado: {model}")

    pipe = MANAGER.get(model, loader_fn)
    width, height = _ratio_to_size(aspect_ratio)
    num_frames = int(round(duration_seconds * FPS))
    num_frames = max(24, min(num_frames, 240))

    out = pipe(
        prompt=prompt,
        num_frames=num_frames,
        height=height,
        width=width,
        guidance_scale=6.0,
    )
    frames = out.frames[0]
    video_bytes = _frames_to_mp4(frames, fps=FPS)
    video_b64 = base64.b64encode(video_bytes).decode("utf-8")
    return {
        "video_b64": video_b64,
        "elapsed_s": round(time.time() - t0, 1),
        "model_used": model,
        "vram_peak_mb": int(torch.cuda.max_memory_allocated() / 1e6),
    }

print("Generation funcs OK")

## Celda 5 · Servidor FastAPI + túnel ngrok

Levanta el servidor HTTP que Guion_expert consulta. Emite la URL pública del túnel para copiar al `.env` como `TRINITY_URL`.

In [ ]:
from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn, threading, torch
from pyngrok import ngrok

app = FastAPI(title="OpenMontage Trinity Pro")

class I2VReq(BaseModel):
    image_b64: str
    prompt: str
    model: str
    duration_seconds: float = 5.0
    aspect_ratio: str = "16:9"

class T2VReq(BaseModel):
    prompt: str
    model: str
    duration_seconds: float = 5.0
    aspect_ratio: str = "16:9"

def _check_auth(request: Request):
    supplied = request.headers.get("x-trinity-token", "")
    if supplied != TRINITY_TOKEN:
        raise HTTPException(status_code=401, detail="invalid X-Trinity-Token")

@app.get("/health")
def health():
    return {
        "status": "ok",
        "current_model": MANAGER.current(),
        "models_registered": list(LOADERS.keys()),
        "vram_mb": MANAGER.vram_mb(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    }

@app.post("/i2v")
def i2v(req: I2VReq, request: Request):
    _check_auth(request)
    try:
        return run_i2v(req.model, req.image_b64, req.prompt,
                       req.duration_seconds, req.aspect_ratio)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")

@app.post("/t2v")
def t2v(req: T2VReq, request: Request):
    _check_auth(request)
    try:
        return run_t2v(req.model, req.prompt,
                       req.duration_seconds, req.aspect_ratio)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"{type(e).__name__}: {e}")

# Launch uvicorn in background thread (permite seguir usando el notebook)
def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Túnel ngrok (instalá NGROK_AUTH_TOKEN como env var o acá)
NGROK_AUTH = os.getenv("NGROK_AUTH_TOKEN", "")
if NGROK_AUTH:
    ngrok.set_auth_token(NGROK_AUTH)
tunnel = ngrok.connect(8000)
print("\n" + "=" * 60)
print("🌐 TRINITY ONLINE")
print("=" * 60)
print(f"TRINITY_URL={tunnel.public_url}")
print(f"TRINITY_TOKEN={TRINITY_TOKEN}")
print("=" * 60)
print("Copiá esas dos líneas al archivo .env de Guion_expert")
print("y seteá TRINITY_ENABLED=true para activar el backend Trinity.")
print("=" * 60)

## Celda 6 · Test local (opcional — antes de conectar Guion_expert)

Verifica que un T2V simple funcione con SkyReels. Útil para calibrar tiempos antes de enchufar el pipeline real.

In [ ]:
import requests, json

r = requests.post(
    "http://127.0.0.1:8000/t2v",
    headers={"X-Trinity-Token": TRINITY_TOKEN},
    json={
        "prompt": "close up of a woman with warm orange rim light, 35mm film grain, golden hour, soft wind moves her hair, cinematic",
        "model": "skyreels-v1",
        "duration_seconds": 4.0,
        "aspect_ratio": "16:9",
    },
    timeout=600,
)
if r.status_code == 200:
    data = r.json()
    print(f"✅ OK — model={data['model_used']} · elapsed={data['elapsed_s']}s · VRAM peak={data['vram_peak_mb']} MB")
    # Guardar test output
    import base64
    with open("/tmp/trinity_test.mp4", "wb") as f:
        f.write(base64.b64decode(data["video_b64"]))
    print("Video guardado en /tmp/trinity_test.mp4")
else:
    print("❌ FAIL", r.status_code, r.text[:300])

## Celda 7 · Guía rápida de integración con Guion_expert

```bash
# 1. Copiar al .env de Guion_expert lo que emitió la Celda 5:
TRINITY_ENABLED=true
TRINITY_URL=https://xxxx.ngrok-free.app
TRINITY_TOKEN=<token hex>

# 2. Correr el pipeline normalmente:
cd /path/to/ESCRIBE
python master_orchestrator.py -i "tu idea"

# 3. El orchestrator detecta Trinity, lo usa para las escenas recomendadas,
#    y cae a fal.ai si Trinity se cae (ChainedProvider).
```

**Para apagar Trinity sin borrar la URL:**
```bash
TRINITY_ENABLED=false  # basta con esto — 100% fal.ai
```

**Para debugear qué backend tomó cada escena:** mirá `logs/` y `stages/scene_plan.json` (cada escena tiene `master_stack.backend` y `master_stack.routing_reason`).